# ChatBot

## Check which python

In [1]:
import sys
print(sys.executable)

d:\Code\swe_agent_jom\.venv\Scripts\python.exe


## load deepseek api

In [2]:
from pathlib import Path
from dotenv import load_dotenv
import os

env_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=env_path)

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
print("DEEPSEEK_API_KEY loaded:", deepseek_api_key is not None)

DEEPSEEK_API_KEY loaded: True


## LLM API Call Example

In [3]:
from openai import OpenAI

client = OpenAI(
    api_key=deepseek_api_key,
    base_url="https://api.deepseek.com",
)

response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[
        {
            "role": "system",
            "content": "You are a research helper. Be accurate and concise"
        },
        {
            "role": "user",
            "content": "Explain what is AI Agent in 3 key points."
        }
    ],
    stream=False,
)

print(response.choices[0].message.content)

1. **Autonomous Decision-Making**: An AI agent operates independently, using its own reasoning to choose actions without constant human input.
2. **Perception and Action**: It perceives its environment (via data or sensors) and acts upon it (e.g., executing tasks, controlling systems) to achieve objectives.
3. **Goal-Oriented Behavior**: The agent is driven by a specific goal or set of objectives, often using tools like LLMs, APIs, or memory to plan and execute multi-step tasks.


## Stream output & temperature

In [4]:
stream  = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[
        {
            "role": "system",
            "content": "You are a rigorous research assistant. Your responses should be well-structured, verifiable, and free from fabricated information."
        },
        {
            "role": "user",
            "content": "Introduce Agent Memory to me with 3 key point."
        }
    ],
    stream=True,
    temperature=0.1 # Smaller lead to more stable and comservative, larger lead to more creative and emanative
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="")

Here are three key points about Agent Memory in the context of AI agents (such as language model–based agents or autonomous systems):

1. **Definition & Purpose**  
   Agent memory is the mechanism that allows an AI agent to store, retrieve, and update information across interactions. It enables the agent to maintain context over time (e.g., remembering user preferences, past actions, or environmental states), moving beyond stateless, single-turn responses.

2. **Key Types**  
   - **Short‑term (working) memory** – holds immediate conversation or task context (e.g., the current dialogue turns or recent observations), typically with limited capacity.  
   - **Long‑term memory** – stores persistent knowledge (e.g., user profiles, learned facts, historical events) using external storage (vector databases, knowledge graphs) or internal model updates.  
   - **Episodic vs. semantic** – some architectures further distinguish memory for specific past experiences (episodic) from general factua

## Structured output

In [5]:
client = OpenAI(
    api_key=deepseek_api_key,
    base_url="https://api.deepseek.com",
)

response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=[
        {"role": "system", "content": "You are a research helper. Be accurate and concise. You can only output valid json object."},
        {"role": "user", "content": "Explain what is AI Agent in 3 key points."}
    ],
    stream=False,
    response_format={"type": "json_object"},
)
print(response.choices[0].message.content)

{
  "points": [
    "An AI agent is an autonomous entity that perceives its environment through sensors and acts upon it using actuators to achieve specific goals.",
    "It utilizes artificial intelligence techniques such as machine learning, reasoning, and planning to make decisions and adapt its behavior based on feedback.",
    "Key characteristics include autonomy, reactivity to the environment, pro-activeness (goal-directed behavior), and social ability (interaction with other agents or humans)."
  ]
}


## Multiple rounds of conversation

In [6]:
from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam
from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv())

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
)

MODEL = "deepseek-v4-flash"  

messages: list[ChatCompletionMessageParam] = [
    {
        "role": "system",
        "content": "You are a rigorous research assistant. Be consice and accurate."
    }
]

def chat(user_input: str) -> str|None:
    messages.append({
        "role": "user",
        "content": user_input
    })

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.2,
    )

    assistant_reply = response.choices[0].message.content

    messages.append({
        "role": "assistant",
        "content": assistant_reply
    })

    return assistant_reply

In [7]:
chat("My name is JOM, I am working on AI Agents.")

'Hello JOM. How can I assist you with your work on AI Agents?'

In [8]:
chat("What is my name? And what I am focusing on?")

'Your name is JOM, and you are focusing on AI Agents.'

In [9]:
chat("How do you know that?")

'I know that because you explicitly stated it in your first message: "My name is JOM, I am working on AI Agents."'

In [10]:
chat("What if the message is too large?")

"If a message or input exceeds the AI's context window (token limit), several strategies are commonly used:\n\n- **Truncation**: The oldest or least relevant parts are cut off, potentially losing important context.\n- **Chunking**: The input is split into smaller segments, processed separately, and results are combined.\n- **Summarization**: Earlier parts are summarized before adding new content.\n- **Sliding window**: Only the most recent N tokens are retained for each new interaction.\n\nIn AI Agents, this is critical for long-running tasks, memory management, and maintaining coherence. The specific handling depends on the system's design—e.g., using external memory or retrieval-augmented generation (RAG) to manage large contexts."

In [12]:
for message in messages:
    role = message["role"]
    content = message.get("content", "")
    print(f"{role}: {content}")
    print()

system: You are a rigorous research assistant. Be consice and accurate.

user: My name is JOM, I am working on AI Agents.

assistant: Hello JOM. How can I assist you with your work on AI Agents?

user: What is my name? And what I am focusing on?

assistant: Your name is JOM, and you are focusing on AI Agents.

user: How do you know that?

assistant: I know that because you explicitly stated it in your first message: "My name is JOM, I am working on AI Agents."

user: What if the message is too large?

assistant: If a message or input exceeds the AI's context window (token limit), several strategies are commonly used:

- **Truncation**: The oldest or least relevant parts are cut off, potentially losing important context.
- **Chunking**: The input is split into smaller segments, processed separately, and results are combined.
- **Summarization**: Earlier parts are summarized before adding new content.
- **Sliding window**: Only the most recent N tokens are retained for each new interacti